# 04 — Gold Star Schema

Customer-facing star schema built on top of silver. Three dimensions, one
wide fact table, plus four pre-aggregated views that drive the dashboard.

```
       dim_artist ─┐
                   │
       dim_company ─── fact_artist_team ─── dim_person
```

| Gold table | Grain | Clustered by |
|------------|-------|--------------|
| `dim_artist`        | artist | — |
| `dim_company`       | company (firm) | `(role)` |
| `dim_person`        | person × company | `(role)` |
| `fact_artist_team`  | artist × company × person × role | `(role)` |

Views:
* `v_agency_market_share` — % of seed artists each agency represents
* `v_top_managers`        — managers ranked by artist count
* `v_artists_by_label`    — labels ranked by artist count
* `v_artist_team_summary` — one row per artist with comma-joined firms (Genie-friendly)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [ ]:
UC_CATALOG    = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA     = os.getenv('UC_SCHEMA',  'rostr_music_industry')
SILVER_SCHEMA = f'{UC_SCHEMA}_silver'
GOLD_SCHEMA   = f'{UC_SCHEMA}_gold'

S = f'{UC_CATALOG}.{SILVER_SCHEMA}'
G = f'{UC_CATALOG}.{GOLD_SCHEMA}'
print(f'Silver: {S}\nGold:   {G}')

## Dimensions

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_artist AS
    SELECT
        artist_sk,
        rostr_id            AS artist_handle,
        artist_name,
        artist_type,
        gender,
        age,
        birth_date,
        location,
        genres,
        spotify_followers,
        instagram_followers,
        tiktok_followers,
        youtube_subscribers,
        bandsintown_on_tour,
        ai_origin_country,
        ai_origin_city,
        ai_career_start,
        ai_roles,
        wikipedia_url,
        official_website,
        profile_url
    FROM {S}.silver_artists
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_company (
        company_sk        STRING NOT NULL,
        company_handle    STRING,
        company_name      STRING,
        role              STRING,
        record_label_type STRING,
        hq_locations      ARRAY<STRING>,
        website_url       STRING,
        ai_year_founded   INT,
        parent_company    STRING
    )
    CLUSTER BY (role)
""")
spark.sql(f"""
    INSERT OVERWRITE {G}.dim_company
    SELECT
        company_sk,
        rostr_id          AS company_handle,
        company_name,
        role,
        record_label_type_fmt,
        hq_locations,
        website_url,
        ai_year_founded,
        parent_company
    FROM {S}.silver_companies
""")

# dim_person is keyed at (person, company) grain so the same individual at two
# different firms gets two rows -- their `role` and `email` differ per firm.
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_person (
        person_sk         STRING NOT NULL,
        person_handle     STRING,
        person_name       STRING,
        role              STRING,
        person_role       STRING,
        company_sk        STRING,
        company_name      STRING,
        person_email      STRING,
        person_phone      STRING,
        exec_roles        ARRAY<STRING>,
        person_profile_url STRING
    )
    CLUSTER BY (role)
""")
spark.sql(f"""
    INSERT OVERWRITE {G}.dim_person
    SELECT DISTINCT
        MD5(CONCAT_WS('||', ap.person_rostr_id, ap.company_rostr_id)) AS person_sk,
        ap.person_rostr_id,
        ap.person_name,
        ap.role,
        ap.person_role,
        MD5(ap.company_rostr_id) AS company_sk,
        ap.company_name,
        ap.person_email,
        ap.person_phone,
        ap.exec_roles,
        ap.person_profile_url
    FROM {S}.silver_artist_person ap
    WHERE ap.person_rostr_id IS NOT NULL AND ap.company_rostr_id IS NOT NULL
""")
print('dimensions built.')
for t in ('dim_artist','dim_company','dim_person'):
    print(f'  {G}.{t}: {spark.table(f"{G}.{t}").count()} rows')

## fact_artist_team

The wide fact joining everything together. Grain: artist × company × person × role.
Each row carries the surrogate keys of all three dims for clean star-join ergonomics.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.fact_artist_team (
        artist_team_sk     STRING NOT NULL,
        artist_sk          STRING,
        artist_handle      STRING,
        company_sk         STRING,
        company_handle     STRING,
        company_name       STRING,
        person_sk          STRING,
        person_handle      STRING,
        person_name        STRING,
        person_role        STRING,
        person_email       STRING,
        person_phone       STRING,
        role               STRING,
        _ingestion_timestamp TIMESTAMP
    )
    CLUSTER BY (role)
""")
spark.sql(f"""
    INSERT OVERWRITE {G}.fact_artist_team
    SELECT
        ap.artist_person_sk                        AS artist_team_sk,
        a.artist_sk,
        ap.artist_handle,
        MD5(ap.company_rostr_id)                   AS company_sk,
        ap.company_rostr_id                        AS company_handle,
        ap.company_name,
        MD5(CONCAT_WS('||', ap.person_rostr_id, ap.company_rostr_id)) AS person_sk,
        ap.person_rostr_id                         AS person_handle,
        ap.person_name,
        ap.person_role,
        ap.person_email,
        ap.person_phone,
        ap.role,
        ap._ingestion_timestamp
    FROM {S}.silver_artist_person ap
    LEFT JOIN {S}.silver_artists a ON a.rostr_id = ap.artist_handle
""")
print('fact_artist_team:', spark.table(f'{G}.fact_artist_team').count())

## Analytical views

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_agency_market_share AS
    WITH agency_artists AS (
        SELECT DISTINCT company_name, artist_handle
        FROM {S}.silver_artist_company
        WHERE role = 'AGENCY' AND company_name IS NOT NULL
    ),
    total AS (
        SELECT COUNT(DISTINCT artist_handle) AS n FROM agency_artists
    )
    SELECT
        a.company_name AS agency,
        COUNT(DISTINCT a.artist_handle)                AS artists_represented,
        ROUND(COUNT(DISTINCT a.artist_handle) * 100.0
              / NULLIF((SELECT n FROM total), 0), 1)   AS market_share_pct
    FROM agency_artists a
    GROUP BY a.company_name
    ORDER BY artists_represented DESC
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_top_managers AS
    SELECT
        person_name,
        company_name,
        COUNT(DISTINCT artist_handle) AS artist_count,
        ARRAY_AGG(DISTINCT artist_handle)             AS represents
    FROM {S}.silver_artist_person
    WHERE role = 'MANAGEMENT' AND person_name IS NOT NULL
    GROUP BY person_name, company_name
    ORDER BY artist_count DESC, person_name
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_artists_by_label AS
    SELECT
        c.company_name              AS label,
        c.record_label_type_fmt     AS label_type,
        COUNT(DISTINCT ac.artist_handle) AS artist_count
    FROM {S}.silver_artist_company ac
    JOIN {S}.silver_companies c ON c.rostr_id = ac.company_rostr_id
    WHERE ac.role = 'RECORD_LABEL'
    GROUP BY c.company_name, c.record_label_type_fmt
    ORDER BY artist_count DESC
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_artist_team_summary AS
    SELECT
        a.artist_name,
        a.artist_handle,
        a.spotify_followers,
        a.instagram_followers,
        CONCAT_WS(' / ', COLLECT_SET(CASE WHEN ac.role = 'AGENCY'        THEN ac.company_name END)) AS agencies,
        CONCAT_WS(' / ', COLLECT_SET(CASE WHEN ac.role = 'MANAGEMENT'    THEN ac.company_name END)) AS management_firms,
        CONCAT_WS(' / ', COLLECT_SET(CASE WHEN ac.role = 'RECORD_LABEL'  THEN ac.company_name END)) AS labels,
        CONCAT_WS(' / ', COLLECT_SET(CASE WHEN ac.role = 'PUBLISHER'     THEN ac.company_name END)) AS publishers
    FROM {G}.dim_artist a
    LEFT JOIN {S}.silver_artist_company ac ON ac.artist_handle = a.artist_handle
    GROUP BY a.artist_name, a.artist_handle, a.spotify_followers, a.instagram_followers
    ORDER BY a.spotify_followers DESC NULLS LAST
""")

print('views built.')

## Verify

In [ ]:
print('=' * 60)
print('GOLD LAYER SUMMARY')
print('=' * 60)
for t in ['dim_artist','dim_company','dim_person','fact_artist_team']:
    full = f'{G}.{t}'
    n = spark.table(full).count()
    print(f'  {full:<70s}  {n:>10,} rows')

print('\nAgency market share (top 10):')
spark.sql(f'SELECT * FROM {G}.v_agency_market_share LIMIT 10').show(truncate=False)

print('Artist team summary (top 10 by Spotify followers):')
spark.sql(f'SELECT * FROM {G}.v_artist_team_summary LIMIT 10').show(truncate=False)